# Imports

In [1]:
from presyo_ph.spark import get_spark_session
from presyo_ph.medallion.bronze import load_bronze
from pyspark.sql import functions as F

# Load spark session and bronze table

In [2]:
spark = get_spark_session()

bronze_df = load_bronze(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/01 20:03:27 WARN Utils: Your hostname, Miggys-TUF-A15.local, resolves to a loopback address: 127.0.0.1; using 10.22.35.70 instead (on interface feth4322)
26/11/01 20:03:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/miggyfortuna/Development/ml/presyo-ph/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/miggyfortuna/.ivy2.5.2/cache
The jars for the packages stored in: /Users/miggyfortuna/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c2c35336-a22e-4e21-bc00-49badf2978bc;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found o

Bronze ingestion completed: local


# Inspection

In [3]:
bronze_df.printSchema()

root
 |-- ISO3: string (nullable = true)
 |-- country: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- mkt_name: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- geo_id: string (nullable = true)
 |-- price_date: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- components: string (nullable = true)
 |-- start_dense_data: string (nullable = true)
 |-- last_survey_point: string (nullable = true)
 |-- data_coverage: string (nullable = true)
 |-- data_coverage_recent: string (nullable = true)
 |-- index_confidence_score: string (nullable = true)
 |-- spatially_interpolated: string (nullable = true)
 |-- beans: string (nullable = true)
 |-- cabbage: string (nullable = true)
 |-- carrots: string (nullable = true)
 |-- eggs: string (nullable = true)
 |-- garlic: string (nullable = true)
 |

In [4]:
(
    bronze_df
    .limit(5)
    .withColumn(
        "_ingested_at",
        F.date_format("_ingested_at", "yyyy-MM-dd HH:mm:ss")
    )
    .toPandas()
)

,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,price_date,year,...,inflation_tomatoes,trust_tomatoes,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index,_ingested_at,_source_file
0,PHL,Philippines,Cordillera Administrative region,Abra,Abra,17.6,120.62,gid_1760000001206200000,2007-01-01,2007,...,None,5.5,0.72,0.74,0.69,0.72,None,9.2,2026-11-01 20:03:38,/Users/miggyfortuna/Development/ml/presyo-ph/d...
1,PHL,Philippines,Cordillera Administrative region,Abra,Abra,17.6,120.62,gid_1760000001206200000,2007-02-01,2007,...,None,5.5,0.72,0.74,0.7,0.7,None,9.2,2026-11-01 20:03:38,/Users/miggyfortuna/Development/ml/presyo-ph/d...
2,PHL,Philippines,Cordillera Administrative region,Abra,Abra,17.6,120.62,gid_1760000001206200000,2007-03-01,2007,...,None,5.5,0.7,0.72,0.68,0.69,None,9.2,2026-11-01 20:03:38,/Users/miggyfortuna/Development/ml/presyo-ph/d...
3,PHL,Philippines,Cordillera Administrative region,Abra,Abra,17.6,120.62,gid_1760000001206200000,2007-04-01,2007,...,None,5.5,0.69,0.71,0.66,0.7,None,9.2,2026-11-01 20:03:38,/Users/miggyfortuna/Development/ml/presyo-ph/d...
4,PHL,Philippines,Cordillera Administrative region,Abra,Abra,17.6,120.62,gid_1760000001206200000,2007-05-01,2007,...,None,5.5,0.71,0.73,0.68,0.7,None,9.2,2026-11-01 20:03:38,/Users/miggyfortuna/Development/ml/presyo-ph/d...


In [5]:
print("Rows:", bronze_df.count())
print("Columns:", len(bronze_df.columns))

Rows: 24743
Columns: 90


In [6]:
bronze_df.select(
    "_ingested_at",
    "_source_file"
).show(5, truncate=False)

+--------------------------+----------------------------------------------------------------------------------+
|_ingested_at              |_source_file                                                                      |
+--------------------------+----------------------------------------------------------------------------------+
|2026-11-01 20:03:38.829148|/Users/miggyfortuna/Development/ml/presyo-ph/data/raw/extracted/food_prices_ph.csv|
|2026-11-01 20:03:38.829148|/Users/miggyfortuna/Development/ml/presyo-ph/data/raw/extracted/food_prices_ph.csv|
|2026-11-01 20:03:38.829148|/Users/miggyfortuna/Development/ml/presyo-ph/data/raw/extracted/food_prices_ph.csv|
|2026-11-01 20:03:38.829148|/Users/miggyfortuna/Development/ml/presyo-ph/data/raw/extracted/food_prices_ph.csv|
|2026-11-01 20:03:38.829148|/Users/miggyfortuna/Development/ml/presyo-ph/data/raw/extracted/food_prices_ph.csv|
+--------------------------+----------------------------------------------------------------------------

# Check Persistence

In [7]:
from presyo_ph.config import BRONZE_PATH

persisted_df = (
    spark.read
    .format("delta")
    .load(str(BRONZE_PATH))
)

print("Persisted rows:", persisted_df.count())

display_df = (
    persisted_df
    .limit(5)
    .withColumn(
        "_ingested_at",
        F.date_format("_ingested_at", "yyyy-MM-dd HH:mm:ss")
    )
    .toPandas()
)

display_df

Persisted rows: 24743


,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,price_date,year,...,inflation_tomatoes,trust_tomatoes,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index,_ingested_at,_source_file
0,PHL,Philippines,Region VI,Iloilo,Iloilo,10.69,122.57,gid_1069000001225700000,2020-09-01,2020,...,-18.13,8.9,1.11,1.14,1.08,1.14,8.43,9.7,2026-11-01 20:03:35,/Users/miggyfortuna/Development/ml/presyo-ph/d...
1,PHL,Philippines,Region VI,Iloilo,Iloilo,10.69,122.57,gid_1069000001225700000,2020-10-01,2020,...,15.57,9.3,1.13,1.25,1.1,1.25,15.82,9.7,2026-11-01 20:03:35,/Users/miggyfortuna/Development/ml/presyo-ph/d...
2,PHL,Philippines,Region VI,Iloilo,Iloilo,10.69,122.57,gid_1069000001225700000,2020-11-01,2020,...,92.02,9.7,1.27,1.44,1.21,1.44,35.37,9.8,2026-11-01 20:03:35,/Users/miggyfortuna/Development/ml/presyo-ph/d...
3,PHL,Philippines,Region VI,Iloilo,Iloilo,10.69,122.57,gid_1069000001225700000,2020-12-01,2020,...,107.32,9.9,1.46,1.55,1.35,1.35,27.99,9.8,2026-11-01 20:03:35,/Users/miggyfortuna/Development/ml/presyo-ph/d...
4,PHL,Philippines,Region VI,Iloilo,Iloilo,10.69,122.57,gid_1069000001225700000,2021-01-01,2021,...,39.25,10,1.24,1.32,1.17,1.2,11.3,9.8,2026-11-01 20:03:35,/Users/miggyfortuna/Development/ml/presyo-ph/d...


In [8]:
null_counts = bronze_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in bronze_df.columns
])

null_counts_pd = null_counts.toPandas()

null_counts_pd.T.rename(
    columns={0: "null_count"}
).sort_values(
    "null_count",
    ascending=False
)

,null_count
beans,21942
potatoes,21732
garlic,21331
cabbage,21285
carrots,21182
...,...
l_cabbage,0
h_cabbage,0
o_cabbage,0
trust_beans,0


In [9]:
bronze_df.select(
    F.min("price_date").alias("earliest_date"),
    F.max("price_date").alias("latest_date")
).show()

+-------------+-----------+
|earliest_date|latest_date|
+-------------+-----------+
|   2007-01-01| 2025-11-01|
+-------------+-----------+



In [10]:
bronze_df.select("mkt_name").distinct().orderBy("mkt_name").show(120, truncate=False)

+------------------------+
|mkt_name                |
+------------------------+
|Abra                    |
|Agusan del Norte        |
|Agusan del Sur          |
|Aklan                   |
|Albay                   |
|Antique                 |
|Apayao                  |
|Aurora                  |
|Baguio City             |
|Basilan                 |
|Bataan                  |
|Batangas                |
|Batangas City           |
|Benguet                 |
|Biliran                 |
|Bohol                   |
|Bukidnon                |
|Bulacan                 |
|Butuan City             |
|Cabanatuan City         |
|Cagayan                 |
|Cagayan de Oro City     |
|Calapan City            |
|Camarines Norte         |
|Camarines Sur           |
|Camiguin                |
|Capiz                   |
|Catanduanes             |
|Catarman                |
|Cavite                  |
|Cebu                    |
|Cebu City               |
|Cotabato                |
|Cotabato City           |
|

In [11]:
bronze_df.select(
    "mkt_name",
    "price_date",
    "currency",
    "rice"
).filter(
    F.col("rice").isNotNull()
).show(30, truncate=False)

+--------+----------+--------+-----+
|mkt_name|price_date|currency|rice |
+--------+----------+--------+-----+
|Abra    |2020-05-01|PHP     |39.66|
|Abra    |2020-06-01|PHP     |40.12|
|Abra    |2020-07-01|PHP     |41.22|
|Abra    |2020-08-01|PHP     |40.31|
|Abra    |2020-09-01|PHP     |40.12|
|Abra    |2020-10-01|PHP     |40.25|
|Abra    |2020-11-01|PHP     |40.08|
|Abra    |2020-12-01|PHP     |38.23|
|Abra    |2021-01-01|PHP     |36.5 |
|Abra    |2021-02-01|PHP     |36.5 |
|Abra    |2021-03-01|PHP     |37.75|
|Abra    |2021-04-01|PHP     |38.25|
|Abra    |2021-05-01|PHP     |38.25|
|Abra    |2021-06-01|PHP     |37.75|
|Abra    |2021-07-01|PHP     |37.75|
|Abra    |2021-08-01|PHP     |38   |
|Abra    |2021-09-01|PHP     |37.67|
|Abra    |2021-10-01|PHP     |37.67|
|Abra    |2021-11-01|PHP     |37.67|
|Abra    |2021-12-01|PHP     |37.67|
|Abra    |2022-01-01|PHP     |37.67|
|Abra    |2022-02-01|PHP     |38   |
|Abra    |2022-03-01|PHP     |38.33|
|Abra    |2022-04-01|PHP     |38.33|
|

# Notes
Bronze table has no data transformations.
_ingested_at and _source_file metadata columns were added.
Dimensions:
    - Rows: 24743
    - Columns: 90 (88 original + 2 metadata)